# **Baseline**

1. Load Data

In [33]:
# import pandas as pd
# df = pd.read_csv("../data/raw/creditcard.csv")

In [34]:
# df = df.sample(n=50000, random_state=42)

In [3]:
# 中文注释：让 notebook 能找到项目根目录下的 src 包
# 如果你 notebook 在 notebooks/ 下，src 在项目根目录，需要把项目根加入路径
import sys
from pathlib import Path
project_root = Path.cwd().parent  # notebook 在 notebooks/ 里，往上一级是项目根
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 中文注释：导入我们刚抽出来的函数
from src.data.load_data import load_creditcard, split_features_and_target, stratified_split
from src.utils.metrics import evaluate


AttributeError: module 'scipy.sparse' has no attribute 'linalg'

In [1]:


df = load_creditcard()

df = df.sample(n=50000, random_state=42)

X, y = df.split_features_and_target()

X_train, X_test, y_train, y_test = stratified_split(X, y, 0.2, 42)

ModuleNotFoundError: No module named 'src'

2. Train_test_split and Pipeline build

In [35]:
# 分层切分数据，保持训练/测试集的欺诈比例一致
# from sklearn.model_selection import train_test_split
# import pandas as pd
import matplotlib.pyplot as plt
# from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBClassifier
# from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, precision_score, recall_score, f1_score


# X = df.drop(columns=["Class"])
# y = df["Class"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42
# )

# 用 Pipeline 把标准化和逻辑回归串起来，避免数据泄漏

# pipe = Pipeline([
#     ("scaler", StandardScaler()),
#     ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
# ])
# pipe.fit(X_train, y_train)

def train_lr(X_train, y_train, X_test, y_test):
    # 搭建流水线：标准化 + 逻辑回归
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ])
    pipe.fit(X_train, y_train)
    # 评估模型
    # model_evaluation(pipe, X_test, y_test)
    return pipe

def train_rf(X_train, y_train, X_test, y_test):
    # 树模型不需要标准化，直接训练
    pipe = Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=100,       # 决策树数量，可自行修改
            class_weight="balanced",# 适配不平衡样本
            random_state=42
        ))
    ])
    pipe.fit(X_train, y_train)
    # model_evaluation(pipe, X_test, y_test)
    return pipe

def train_xgb(X_train, y_train, X_test, y_test):
    pipe = Pipeline([
        ("clf", XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,      # 学习率，可自行调参
            scale_pos_weight=10,     # 不平衡样本权重，根据正负比例修改
            random_state=42,
            use_label_encoder=False,
            eval_metric="logloss"
        ))
    ])
    pipe.fit(X_train, y_train)
    # model_evaluation(pipe, X_test, y_test)
    return pipe



3.Model Training

In [ ]:
# 训练模型
pipe_lr = train_lr(X_train, y_train, X_test, y_test)

pipe_rf = train_rf(X_train, y_train, X_test, y_test)

pipe_xgb = train_xgb(X_train, y_train, X_test, y_test)

4.Find best thresholds when considering business logic

In [37]:
# 中文注释：定义业务成本，寻找最优阈值
import numpy as np

def find_best_t(model, fn_cost=500, fp_cost=5): # Business logic
    proba = model.predict_proba(X_test)[:, 1]
    thresholds = np.linspace(0.01, 0.99, 99)
    costs = []
    for t in thresholds:
        y_pred = (proba > t).astype(int)
        fn = ((y_pred == 0) & (y_test == 1)).sum()  # 漏判欺诈
        fp = ((y_pred == 1) & (y_test == 0)).sum()  # 误报
        cost = fn_cost * fn + fp_cost * fp                    
        costs.append(cost)

    best_t = thresholds[np.argmin(costs)]
    # print("Best thresholds:", best_t)
    
    def plot_threshold_cost():
        # 绘制 阈值(x) - 业务成本(y) 曲线
        plt.figure(figsize=(10, 6))
        plt.plot(thresholds, costs, color="steelblue", linewidth=2, label="Business Cost")

        # 标注最优阈值点（成本最低）
        plt.scatter(best_t, min(costs), color="crimson", s=80, zorder=5, label="Optimal Threshold")

        # 辅助线 + 标注文字
        plt.axvline(x=best_t, color="crimson", linestyle="--", alpha=0.7)
        plt.title("Threshold vs. Business Cost", fontsize=14)
        plt.xlabel("Classification Threshold", fontsize=12)
        plt.ylabel(f"Total Business Cost ({fn_cost}*FN + {fp_cost}*FP)", fontsize=12)
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()

        clf = model.steps[-1][1]
        auto_name = clf.__class__.__name__
        # 打印最优结果
        print(f"Best Threshold for {auto_name}：{best_t:.4f}")
        print(f"Minimized Cost for {auto_name}：{min(costs)}")
    
    # 挂载绘图函数到主函数，外部可调用
    find_best_t.plot = plot_threshold_cost

    return best_t


# proba = pipe_xgb.predict_proba(X_test)[:, 1]
# thresholds = np.linspace(0.01, 0.99, 99)
# costs = []
# for t in thresholds:
#     y_pred = (proba > t).astype(int)
#     fn = ((y_pred == 0) & (y_test == 1)).sum()  # 漏判欺诈
#     fp = ((y_pred == 1) & (y_test == 0)).sum()  # 误报
#     cost = 500 * fn + 5 * fp
#     costs.append(cost)

# best_t = thresholds[np.argmin(costs)]
# print("Best thresholds:", best_t)

5.Evaluate each model

In [ ]:
# 输出关键指标——因为样本极度不平衡，重点看 PR-AUC 和 Recall
# y_proba = pipe.predict_proba(X_test)[:, 1]
# print("ROC-AUC:", roc_auc_score(y_test, y_proba))
# print("PR-AUC :", average_precision_score(y_test, y_proba))
# print(classification_report(y_test, y_proba > 0.5))

# 终极升级版：包含全部核心指标（适配不平衡数据，取正类1的指标）
# def evaluate(name, model, X_test, y_test, best_t):
    # proba = model.predict_proba(X_test)[:, 1]
    # y_pred = proba > best_t

    # return {
    #     "model": name,
    #     "ROC-AUC": round(roc_auc_score(y_test, proba), 4),
    #     "PR-AUC": round(average_precision_score(y_test, proba), 4),
    #     # 只统计少数类/正类=1的精准率、召回率、F1（不平衡数据核心指标）
    #     "Precision": round(precision_score(y_test, y_pred, pos_label=1), 4),
    #     "Recall": round(recall_score(y_test, y_pred, pos_label=1), 4),
    #     "F1-Score": round(f1_score(y_test, y_pred, pos_label=1), 4)
    # }

# 批量存储结果
results = []

# 录入指标
results.append(evaluate("Logistic Regression", pipe_lr, X_test, y_test,find_best_t(pipe_lr)))
find_best_t.plot()
results.append(evaluate("Random Forest", pipe_rf, X_test, y_test,find_best_t(pipe_rf)))
find_best_t.plot()
results.append(evaluate("XGBoost", pipe_xgb, X_test, y_test,find_best_t(pipe_xgb)))
find_best_t.plot()

print("")

# 生成完整对比表格
result_df = pd.DataFrame(results)
print("===== Comparison between 3 models =====")
print(result_df)

# best_t_lr = find_best_t(pipe_lr)
# find_best_t.plot()

# best_t_rf = find_best_t(pipe_rf)
# find_best_t.plot()

# best_t_xgb = find_best_t(pipe_xgb)
# find_best_t.plot()


6. Explain the model using SHAP

In [39]:
import shap

def model_shap_explainer(pipeline, X_test, sample_num=200):
    X_sample = X_test.sample(sample_num, random_state=42)

    # 解决新版sklearn Pipeline无transform报错
    # 手动拆分预处理 + 模型，不调用 pipeline.transform
    # pipeline.steps = [
    # ("scaler", StandardScaler()),
    # ("onehot", OneHotEncoder()),
    # ("classifier", XGBClassifier())
    # ]
    pre_steps = pipeline.steps[:-1] #取最后一个之前的所有
    clf = pipeline.steps[-1][1]

    # 手动执行预处理，百分百兼容所有sklearn版本
    X_transformed = X_sample.copy()
    for name, step in pre_steps:
        X_transformed = step.transform(X_transformed) 
        # 读取对象内部已经存好的均值、标准差，对新样本做缩放。 没有读取旧训练数据集，只是复用训练得到的统计参数。

    # 安全获取特征名
    try:
        # 拼接预处理管道获取特征名
        # `pre_steps`只是普通 Python `list`，list 没有任何 sklearn 方法， `.get_feature_names_out()` 是**Pipeline / 转换器对象的实例方法**
        # from sklearn.pipeline import Pipeline
        temp_pre = Pipeline(pre_steps)
        feature_names = temp_pre.get_feature_names_out()
    except Exception:
        feature_names = [f"feat_{i}" for i in range(X_transformed.shape[1])]

    # 选择对应解释器
    clf_type = str(type(clf))
    if "XGB" in clf_type or "RandomForest" in clf_type:
        explainer = shap.TreeExplainer(clf)
    else:
        explainer = shap.LinearExplainer(clf, X_transformed)

    # 彻底根治 IndexError 索引越界
    shap_values = explainer.shap_values(X_transformed)
    
    # 兼容所有SHAP返回格式
    if isinstance(shap_values, list) and len(shap_values) == 2:
        shap_values = shap_values[1]
    elif len(shap_values.shape) == 3:
        shap_values = shap_values[:, :, 1]
    
    # SHAP 有两套返回格式：
    # - 旧版：`[neg_shap, pos_shap]`
    # - 新版：3D 数组 `(sample, feature, class)`
    # 如果是 list 并且长度等于 2，`shap_values[1]`拿到正类全部样本‑特征矩阵；
    # 如果是三维数组，`shap_values[:, :, 1]`：**取全部样本、全部特征，只保留类别 = 1 那一薄层**，压缩回二维矩阵。

    return shap_values, X_transformed, feature_names, explainer

def plot_shap_all(shap_values, X_transformed, feature_names):
    plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False

    # 全局影响+正负趋势图
    plt.figure(figsize=(10,6))
    shap.summary_plot(shap_values, X_transformed, feature_names=feature_names)
    # 纯特征重要性柱状图
    plt.figure(figsize=(10,6))
    shap.summary_plot(shap_values, X_transformed, feature_names=feature_names, plot_type="bar")


In [ ]:
# ========== Function call ==========
shap_vals_lr, X_trans_lr, feat_names_lr, explainer_lr = model_shap_explainer(pipe_lr, X_test, sample_num=300)
shap_vals_rf, X_trans_rf, feat_names_rf, explainer_rf = model_shap_explainer(pipe_rf, X_test, sample_num=300)
shap_vals_xgb, X_trans_xgb, feat_names_xgb, explainer_xgb = model_shap_explainer(pipe_xgb, X_test, sample_num=300)

# 输出两张核心报告图
plot_shap_all(shap_vals_lr, X_trans_lr, feat_names_lr)
plot_shap_all(shap_vals_rf, X_trans_rf, feat_names_rf)
plot_shap_all(shap_vals_xgb, X_trans_xgb, feat_names_xgb)


In [ ]:
# 解释LR第一个样本为什么被预测为欺诈/正常
shap.force_plot(
    explainer_lr.expected_value, 
    shap_vals_lr[0,:], 
    features=X_trans_lr[0,:].round(4),
    feature_names=feat_names_lr,
    matplotlib=True
)
